In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import StackingClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. CHUẨN BỊ DỮ LIỆU
print("🔍 CHUẨN BỊ DỮ LIỆU...")
df = pd.read_excel("validation_data.xlsx")

# Chọn features và target
features = ['BASE_BAL', 'CURR_BAL', 'DUNO_QD', 'LAISUAT', 'LOAIKH']
df['SEX'] = df['SEX'].fillna('UNKNOWN')
sex_mapping = {'ONG': 0, 'BA': 1, 'MR': 0, 'MRS': 1, 'MS': 1, 'UNKNOWN': 2}
df['SEX_ENCODED'] = df['SEX'].map(sex_mapping)
features.append('SEX_ENCODED')

# Làm sạch dữ liệu
df_clean = df[features + ['NHOMNO']].dropna()
df_clean = df_clean[df_clean['NHOMNO'].isin([1, 2, 3, 4, 5])]

X = df_clean[features]
y = df_clean['NHOMNO']

print(f"Kích thước dữ liệu: {X.shape}")
print(f"Phân phối target: {y.value_counts().sort_index().to_dict()}")

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# 2. ĐỊNH NGHĨA CÁC MÔ HÌNH
print("\n" + "="*60)
print("🏗️ ĐỊNH NGHĨA CÁC MÔ HÌNH")
print("="*60)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Neural Network
nn_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate='adaptive',
    max_iter=500,
    random_state=42
)

# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=10,
    num_leaves=31,
    random_state=42
)

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Danh sách các mô hình base
base_models = [
    ('random_forest', rf_model),
    ('neural_network', nn_model),
    ('lightgbm', lgb_model),
    ('xgboost', xgb_model)
]

# Meta-learner (có thể dùng Logistic Regression hoặc model khác)
meta_learner = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0,
    random_state=42,
    max_iter=1000
)

# Stacking model
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,
    passthrough=False,
    n_jobs=-1
)

# 3. HUẤN LUYỆN VÀ ĐÁNH GIÁ TỪNG MÔ HÌNH
print("\n" + "="*60)
print("📊 HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH")
print("="*60)

models = {
    'Random Forest': rf_model,
    'Neural Network': nn_model,
    'LightGBM': lgb_model,
    'XGBoost': xgb_model,
    'Stacking Ensemble': stacking_model
}

results = {}
cv_scores = {}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n🔄 Đang huấn luyện {name}...")
    
    # Cross-validation
    cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_scores[name] = cv_score
    
    # Huấn luyện trên toàn bộ train set
    model.fit(X_train, y_train)
    
    # Dự đoán
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    
    # Tính metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'f1_score': f1,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1-Score: {f1:.4f}")
    print(f"   Cross-val Accuracy: {cv_score.mean():.4f} (+/- {cv_score.std() * 2:.4f})")

# 4. SO SÁNH CHI TIẾT KẾT QUẢ
print("\n" + "="*60)
print("📈 SO SÁNH CHI TIẾT KẾT QUẢ")
print("="*60)

# Tạo bảng so sánh
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[name]['accuracy'] for name in results.keys()],
    'Test F1-Score': [results[name]['f1_score'] for name in results.keys()],
    'CV Mean Accuracy': [cv_scores[name].mean() for name in results.keys()],
    'CV Std': [cv_scores[name].std() for name in results.keys()]
}).sort_values('Test Accuracy', ascending=False)

print("\n📊 BẢNG SO SÁNH HIỆU SUẤT:")
print(comparison_df.round(4))

# 5. TRỰC QUAN HÓA KẾT QUẢ
print("\n" + "="*60)
print("🎨 TRỰC QUAN HÓA KẾT QUẢ")
print("="*60)

plt.figure(figsize=(20, 15))

# Biểu đồ 1: So sánh Accuracy
plt.subplot(2, 3, 1)
models_names = comparison_df['Model'].values
accuracies = comparison_df['Test Accuracy'].values
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFD700', '#FF69B4']

bars = plt.bar(models_names, accuracies, color=colors, alpha=0.8)
plt.title('So sánh Độ chính xác (Accuracy) giữa các Mô hình', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45)
plt.ylim(0, 1.0)

# Thêm giá trị trên mỗi cột
for bar, accuracy in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{accuracy:.4f}', ha='center', va='bottom', fontweight='bold')

# Biểu đồ 2: So sánh F1-Score
plt.subplot(2, 3, 2)
f1_scores = comparison_df['Test F1-Score'].values
bars = plt.bar(models_names, f1_scores, color=colors, alpha=0.8)
plt.title('So sánh F1-Score giữa các Mô hình', fontsize=14, fontweight='bold')
plt.ylabel('F1-Score', fontsize=12)
plt.xticks(rotation=45)
plt.ylim(0, 1.0)

for bar, f1 in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{f1:.4f}', ha='center', va='bottom', fontweight='bold')

# Biểu đồ 3: Cross-validation scores
plt.subplot(2, 3, 3)
box_data = [cv_scores[name] for name in models_names]
box_plot = plt.boxplot(box_data, labels=models_names, patch_artist=True)
plt.title('Phân phối Cross-validation Scores', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45)

# Tô màu cho boxplot
for patch, color in zip(box_plot['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Biểu đồ 4: Confusion Matrix cho Stacking Ensemble
plt.subplot(2, 3, 4)
stacking_pred = results['Stacking Ensemble']['predictions']
cm = confusion_matrix(y_test, stacking_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.title('Confusion Matrix - Stacking Ensemble', fontsize=14, fontweight='bold')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')

# Biểu đồ 5: Feature Importance từ Random Forest (nếu có)
plt.subplot(2, 3, 5)
if hasattr(rf_model, 'feature_importances_'):
    feature_importance = rf_model.feature_importances_
    feature_df = pd.DataFrame({
        'feature': features,
        'importance': feature_importance
    }).sort_values('importance', ascending=True)
    
    plt.barh(feature_df['feature'], feature_df['importance'])
    plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
    plt.xlabel('Importance')

# Biểu đồ 6: Learning Curve so sánh (đơn giản)
plt.subplot(2, 3, 6)
train_sizes = np.linspace(0.1, 1.0, 5)
final_accuracies = [results[name]['accuracy'] for name in models_names]

plt.plot(models_names, final_accuracies, 'o-', linewidth=2, markersize=8)
plt.title('Độ chính xác cuối cùng trên Test Set', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

for i, (model, acc) in enumerate(zip(models_names, final_accuracies)):
    plt.annotate(f'{acc:.4f}', (model, acc), textcoords="offset points", 
                 xytext=(0,10), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# 6. PHÂN TÍCH CHI TIẾT STACKING ENSEMBLE
print("\n" + "="*60)
print("🔍 PHÂN TÍCH CHI TIẾT STACKING ENSEMBLE")
print("="*60)

# Hiển thị classification report chi tiết
print("\n📋 CLASSIFICATION REPORT - STACKING ENSEMBLE:")
stacking_pred = results['Stacking Ensemble']['predictions']
print(classification_report(y_test, stacking_pred, target_names=[f'NHOMNO_{i}' for i in sorted(y.unique())]))

# Phân tích cải thiện so với model tốt nhất đơn lẻ
best_individual_model = comparison_df.iloc[1]['Model']  # Vì stacking luôn tốt nhất
best_individual_accuracy = comparison_df[comparison_df['Model'] == best_individual_model]['Test Accuracy'].values[0]
stacking_accuracy = comparison_df[comparison_df['Model'] == 'Stacking Ensemble']['Test Accuracy'].values[0]

improvement = stacking_accuracy - best_individual_accuracy
improvement_percent = (improvement / best_individual_accuracy) * 100

print(f"\n💪 CẢI THIỆN CỦA STACKING SO VỚI MÔ HÌNH TỐT NHẤT ĐƠN LẺ:")
print(f"   Mô hình đơn lẻ tốt nhất: {best_individual_model}")
print(f"   Độ chính xác {best_individual_model}: {best_individual_accuracy:.4f}")
print(f"   Độ chính xác Stacking: {stacking_accuracy:.4f}")
print(f"   Cải thiện tuyệt đối: {improvement:.4f}")
print(f"   Cải thiện tương đối: {improvement_percent:.2f}%")

# 7. DỰ ĐOÁN VÍ DỤ VỚI STACKING
print("\n" + "="*60)
print("🎯 DỰ ĐOÁN VÍ DỤ VỚI STACKING ENSEMBLE")
print("="*60)

# Lấy 5 mẫu ngẫu nhiên từ test set để minh họa
sample_indices = np.random.choice(len(X_test), 5, replace=False)

print("\nVí dụ dự đoán trên 5 mẫu ngẫu nhiên:")
print("Index | Thực tế | Dự đoán | Đúng/Sai")
print("-" * 40)

for idx in sample_indices:
    actual = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]
    predicted = stacking_pred[idx]
    correct = "✓" if actual == predicted else "✗"
    
    print(f"{idx:5d} | {actual:8d} | {predicted:8d} | {correct}")

# 8. KẾT LUẬN
print("\n" + "="*60)
print("🏆 KẾT LUẬN")
print("="*60)

print("""
📊 TỔNG KẾT SO SÁNH MÔ HÌNH:

✅ ƯU ĐIỂM STACKING ENSEMBLE:
   - Kết hợp điểm mạnh của nhiều mô hình
   - Giảm phương sai và bias
   - Thường cho kết quả tốt hơn mô hình đơn lẻ
   - Linh hoạt với nhiều loại thuật toán

❌ NHƯỢC ĐIỂM:
   - Phức tạp hơn trong huấn luyện
   - Tốn thời gian và tài nguyên tính toán
   - Khó giải thích hơn mô hình đơn lẻ

🎯 ỨNG DỤNG THỰC TẾ:
   - Stacking phù hợp cho các bài toán quan trọng cần độ chính xác cao
   - Có thể sử dụng trong hệ thống xếp hạng tín dụng
   - Phù hợp cho dự đoán rủi ro trong ngân hàng

💡 KHUYẾN NGHỊ:
   - Sử dụng Stacking khi độ chính xác là ưu tiên hàng đầu
   - Sử dụng mô hình đơn lẻ (LightGBM/XGBoost) khi cần cân bằng giữa hiệu suất và tốc độ
   - Random Forest tốt cho việc giải thích feature importance
""")

Columns loaded: ['MJACCTTYPCD', 'PHUONG THUC CHO VAY', 'LOAIKH', 'SEX', 'BASE_BAL', 'CURR_BAL', 'DUNO_QD', 'CURRENCYCD', 'OPEN_DATE', 'NGAYDENHAN', 'ID_TIME', 'DESC_TIME', 'MJACCTTYPDESC', 'ORGNBR', 'ORGNAME', 'PARENTORGNBR', 'PARENTORGNAME', 'LAISUAT', 'MUCDICHVAY', 'NHOMNO', 'NHOMNOMOI', 'NHOMNO_TCBS']
Using target column: NHOMNO
Baseline overall AUC: 0.9993325035633269

Baseline fairness for SEX: dp_diff=0.0485, disparate_impact=0.6714, TPR_diff=0.0769

Baseline fairness for ORGNBR: dp_diff=0.4634, disparate_impact=0.0000, TPR_diff=0.2500
Reweighing model overall AUC: 0.9992742150946553
Reweighing for SEX: dp_diff=0.0365, disparate_impact=0.7302, TPR_diff=0.1171
Reweighing for ORGNBR: dp_diff=0.4634, disparate_impact=0.0000, TPR_diff=0.5000
Postprocess (threshold) for SEX: dp_diff=0.0507, disparate_impact=0.6591, TPR_diff=0.0235
Postprocess (threshold) for ORGNBR: dp_diff=0.4634, disparate_impact=0.0000, TPR_diff=0.2500

Summary table (lower dp_diff/tpr_diff better; auc closer to ba